# Capítulo 4 — Persistência de Dados, Expressões Regulares e Tratamento de Erros

**Programação com Python Aplicada à Engenharia de Defesa** · IPETEC/UCP

> **Notebook de aula — Módulo II, Sábado 2.** Ao final, o miniprojeto integrador ganhará **memória entre execuções** (Módulo 2).

---

### Objetivos do capítulo
Ao final, você será capaz de:
- ler e gravar **arquivos de texto**, usando o bloco `with` para manejo seguro de recursos;
- tratar erros de forma controlada com **exceções** (`try`/`except`), construindo software robusto;
- persistir dados tabulares em **CSV** e dados estruturados em **JSON**;
- armazenar e consultar dados em um banco **SQLite**;
- extrair informação de texto com **expressões regulares**;
- dar **persistência** ao miniprojeto, gravando e recuperando as ocorrências de forma confiável.

*Ao final do Capítulo 3, o miniprojeto modelava ocorrências em memória — com uma fragilidade fatal: ao encerrar o programa, tudo se perdia. Este capítulo resolve esse problema e, no caminho, ensina a lidar com a realidade desordenada dos dados do mundo: arquivos que podem não existir, textos que precisam ser interpretados, valores que vêm malformados.*

> 💡 **Dica de uso no Colab** — Este capítulo **cria arquivos** (`.txt`, `.csv`, `.json`, `.db`). No Google Colab eles vão para a pasta `/content`, visível no ícone de pasta na barra lateral esquerda. Esses arquivos são temporários: somem quando a sessão é encerrada. Execute as células **na ordem**, pois algumas leem o que as anteriores gravaram.

## 4.1 Arquivos de texto

Um arquivo é a forma mais elementar de fazer um dado **sobreviver ao fim do programa**. Trabalhar com arquivos envolve três passos: *abrir*, *ler ou escrever* e *fechar*. A função `open` recebe o nome do arquivo e o **modo** de abertura.

| Modo | Significado |
|---|---|
| `"r"` | leitura (*read*); o arquivo deve existir — é o padrão |
| `"w"` | escrita (*write*); **cria ou sobrescreve** o arquivo |
| `"a"` | acréscimo (*append*); escreve ao final, preservando o conteúdo |

### 4.1.1 Escrevendo e o bloco `with`
A forma recomendada é o bloco `with`, que garante o **fechamento automático** do arquivo — mesmo que ocorra um erro no meio do caminho.

**Listagem 4.1 — Escrevendo um arquivo de texto.**

In [ ]:
linhas = [
    "Radar-A1;44.4;alerta",
    "Radar-B2;50.0;alerta",
    "Sonar-1;18.5;normal",
]

with open("relatorio.txt", "w", encoding="utf-8") as arquivo:
    for linha in linhas:
        arquivo.write(linha + "\n")   # \n insere quebra de linha

print("Relatório gravado.")

O parâmetro `encoding="utf-8"` merece destaque: ele garante que acentos e caracteres especiais sejam gravados corretamente. **Sempre** especifique a codificação ao abrir arquivos de texto — omiti-la é fonte de erros difíceis de diagnosticar, sobretudo entre sistemas operacionais diferentes.

### 4.1.2 Lendo
Para ler, abre-se o arquivo em modo `"r"` (o padrão). É possível ler tudo de uma vez ou, mais comumente, percorrer **linha a linha**.

**Listagem 4.2 — Lendo um arquivo linha a linha.**

In [ ]:
with open("relatorio.txt", "r", encoding="utf-8") as arquivo:
    for linha in arquivo:
        linha = linha.strip()        # remove espaços e a quebra de linha
        print("Lido:", linha)

O método `strip` remove espaços em branco e a quebra de linha do início e do fim — cuidado quase sempre necessário, pois a quebra vem "grudada" no final de cada linha lida.

Vale comparar os modos `"w"` e `"a"` na prática: um sobrescreve, o outro acrescenta.

**Listagem 4.3 — A diferença entre `"w"` e `"a"`.**

In [ ]:
# "w" cria ou SOBRESCREVE
with open("teste_modo.txt", "w", encoding="utf-8") as f:
    f.write("primeira gravação\n")

# "a" ACRESCENTA ao final, preservando o que havia
with open("teste_modo.txt", "a", encoding="utf-8") as f:
    f.write("segunda gravação (acrescentada)\n")

with open("teste_modo.txt", "r", encoding="utf-8") as f:
    print(f.read())

> ✅ **Boa prática** — Prefira sempre o bloco `with` a abrir e fechar arquivos manualmente com `open` e `close`. Além de mais enxuto, ele assegura que o arquivo seja fechado mesmo diante de um erro — evitando vazamento de recursos e arquivos corrompidos. Em sistemas que rodam por longos períodos, como serviços de monitoramento, esse cuidado deixa de ser detalhe e vira **requisito de confiabilidade**.

## 4.2 Tratamento de exceções

Assim que um programa toca o mundo real — arquivos, redes, entradas de usuário —, ele encontra situações que fogem ao seu controle: o arquivo não existe, o disco está cheio, o texto não é um número. Quando isso acontece, o Python lança uma **exceção**; se ninguém a trata, o programa **quebra**. Tratar exceções é o que separa um protótipo de um software robusto.

### 4.2.1 `try` e `except`
O bloco `try` delimita o código que *pode* falhar; o `except` captura a falha e decide o que fazer.

**Listagem 4.4 — Capturando um erro de arquivo inexistente.**

In [ ]:
try:
    with open("inexistente.txt", "r", encoding="utf-8") as arquivo:
        conteudo = arquivo.read()
except FileNotFoundError:
    print("Arquivo não encontrado. Verifique o nome e o caminho.")

Em vez de interromper o programa com uma mensagem técnica, capturamos o erro **específico** e respondemos de forma controlada. Cada tipo de erro tem um nome: `ValueError` (conversão inválida), `KeyError` (chave inexistente em dicionário), `IndexError` (índice fora da lista), entre muitos outros.

**Listagem 4.5 — Tratando entrada inválida do usuário.**

In [ ]:
texto = input("Velocidade em nós: ")
try:
    velocidade = float(texto)
    print(f"Em km/h: {velocidade * 1.852:.1f}")
except ValueError:
    print(f"'{texto}' não é um número válido.")

> 💡 **No Colab** — A célula acima usa `input` e fica **aguardando** você digitar na caixa que aparece logo abaixo dela. Experimente executá-la duas vezes: uma digitando `24`, outra digitando `vinte e quatro`.

### 4.2.2 `else` e `finally`
Dois blocos complementam o `try`: o `else`, executado apenas se **não** houve erro, e o `finally`, executado **sempre** — ideal para ações de limpeza.

**Listagem 4.6 — A estrutura completa do tratamento de erros.**

In [ ]:
def processar(texto):
    try:
        valor = float(texto)
    except ValueError:
        print("Leitura inválida.")
    else:
        print(f"Leitura registrada: {valor}")     # só se deu certo
    finally:
        print("Fim do processamento da leitura.")  # sempre

processar("44.4")
print("---")
processar("erro-de-sensor")

> 🛡️ **Contexto de defesa** — Em um sistema de defesa, a forma como o software **falha** é tão importante quanto a forma como funciona. Um programa de monitoramento que trava ao receber uma leitura malformada de um sensor pode deixar um operador às cegas no pior momento. O tratamento de exceções permite que o sistema registre o problema, descarte o dado ruim e **continue operando** — o princípio da *degradação graciosa*, em que uma falha parcial não derruba o todo.

> ⚠️ **Armadilha comum** — Evite capturar exceções de forma genérica com um `except:` vazio, que "engole" qualquer erro — inclusive os que você nem imaginava, mascarando bugs reais. Capture o tipo **específico** que você sabe tratar (`except ValueError:`). Se um erro inesperado surgir, é melhor que ele apareça do que desapareça silenciosamente.

## 4.3 Dados tabulares: o formato CSV

Muitos dados de engenharia são naturalmente **tabulares** — linhas e colunas, como em uma planilha. O formato **CSV** (*Comma-Separated Values*) é o padrão universal: um arquivo de texto em que cada linha é um registro e os campos são separados por um delimitador. O Python traz o módulo `csv` na biblioteca padrão.

**Listagem 4.7 — Gravando uma lista de dicionários em CSV.**

In [ ]:
import csv

ocorrencias = [
    {"id": 1, "sensor": "Radar-A1", "velocidade_kmh": 44.4},
    {"id": 2, "sensor": "Radar-B2", "velocidade_kmh": 50.0},
]

campos = ["id", "sensor", "velocidade_kmh"]
with open("ocorrencias.csv", "w", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(f, fieldnames=campos)
    escritor.writeheader()              # grava a linha de cabeçalho
    escritor.writerows(ocorrencias)     # grava todos os registros

print("CSV gravado. Conteúdo bruto do arquivo:")
print(open("ocorrencias.csv", encoding="utf-8").read())

A leitura é simétrica: o `csv.DictReader` devolve cada linha já como um **dicionário**, usando o cabeçalho como chaves.

**Listagem 4.8 — Lendo um CSV como dicionários.**

In [ ]:
import csv

with open("ocorrencias.csv", "r", newline="", encoding="utf-8") as f:
    leitor = csv.DictReader(f)
    for linha in leitor:
        # Atenção: todo valor vem como texto!
        vel = float(linha["velocidade_kmh"])
        print(f"{linha['sensor']}: {vel} km/h")

> ⚠️ **Armadilha comum** — Todo valor lido de um CSV chega como **texto** — exatamente como em `input`. O número `44.4` vem como a cadeia `"44.4"`. Esquecer de converter para `float` ou `int` antes de fazer contas é um dos erros mais frequentes ao processar arquivos tabulares. A célula abaixo demonstra o sintoma.

In [ ]:
import csv

with open("ocorrencias.csv", "r", newline="", encoding="utf-8") as f:
    primeira = next(csv.DictReader(f))

bruto = primeira["velocidade_kmh"]
print(type(bruto), repr(bruto))
print("Sem converter, '+' concatena:", bruto + bruto)      # texto!
print("Convertendo, '+' soma:      ", float(bruto) + float(bruto))

## 4.4 Dados estruturados: o formato JSON

Nem todo dado é tabular. Uma ocorrência pode ter campos aninhados, listas, valores opcionais — uma estrutura que não cabe bem em uma tabela plana. Para esses casos, o formato **JSON** (*JavaScript Object Notation*) é a escolha natural: representa, em texto, exatamente as estruturas que já usamos — dicionários, listas, números, textos e valores lógicos. É o formato ideal para persistir a lista de dicionários do miniprojeto.

**Listagem 4.9 — Gravando e lendo JSON.**

In [ ]:
import json

ocorrencias = [
    {"id": 1, "sensor": "Radar-A1", "velocidade_kmh": 44.4, "alerta": True},
    {"id": 2, "sensor": "Radar-B2", "velocidade_kmh": 50.0, "alerta": True},
]

# Gravar (serializar): da estrutura Python para o arquivo
with open("ocorrencias.json", "w", encoding="utf-8") as f:
    json.dump(ocorrencias, f, ensure_ascii=False, indent=2)

# Ler (desserializar): do arquivo para a estrutura Python
with open("ocorrencias.json", "r", encoding="utf-8") as f:
    dados = json.load(f)

print(f"Recuperadas {len(dados)} ocorrências.")
print(dados[0]["sensor"])    # Radar-A1

Os parâmetros de `json.dump` valem nota: `ensure_ascii=False` preserva os acentos legíveis (em vez de convertê-los em códigos de escape), e `indent=2` formata o arquivo de modo indentado e legível por humanos — útil quando alguém precisa inspecioná-lo. Veja a diferença:

**Listagem 4.10 — O efeito de `ensure_ascii` e `indent`.**

In [ ]:
import json

registro = {"sensor": "Radar-A1", "situacao": "detecção não confirmada"}

print("Padrão (ensure_ascii=True, sem indent):")
print(json.dumps(registro))

print("\nLegível (ensure_ascii=False, indent=2):")
print(json.dumps(registro, ensure_ascii=False, indent=2))

> 📝 **Nota** — O JSON conhece listas, mas **não tuplas**. Ao gravar, uma tupla Python — como a nossa `posicao = (-22.9, -43.2)` — é convertida em lista, e assim retorna na leitura: `[-22.9, -43.2]`. Se a distinção importa para o seu código, reconverta após a leitura. É um exemplo de como nenhum formato de persistência preserva *todas* as nuances dos tipos da linguagem.

In [ ]:
import json

posicao = (-22.9, -43.2)                       # tupla
texto = json.dumps({"posicao": posicao})
volta = json.loads(texto)

print(texto)
print(type(volta["posicao"]))                  # <class 'list'> — virou lista!
print(tuple(volta["posicao"]))                 # reconvertendo, se necessário

## 4.5 Banco de dados: SQLite

Arquivos servem bem a volumes modestos. Mas quando o volume cresce, ou quando precisamos de **consultas** eficientes — *"todas as ocorrências do Radar-A1 acima de 40 km/h, ordenadas por velocidade"* —, um banco de dados é a ferramenta certa. O Python traz, de fábrica, o **SQLite**: um banco completo que vive em **um único arquivo**, sem necessidade de servidor. Ideal para aplicações locais como o nosso sistema de apoio à decisão.

A comunicação se dá pela linguagem **SQL**. Vejamos o ciclo completo — criar a tabela, inserir e consultar.

**Listagem 4.11 — Criando, inserindo e consultando com SQLite.**

In [ ]:
import sqlite3

# Conecta (cria o arquivo se não existir)
con = sqlite3.connect("ocorrencias.db")
cur = con.cursor()

# Cria a tabela, se ainda não existir
cur.execute("""
    CREATE TABLE IF NOT EXISTS ocorrencias (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        sensor TEXT,
        velocidade_kmh REAL
    )
""")

# Insere um registro (os ? são preenchidos com segurança)
cur.execute(
    "INSERT INTO ocorrencias (sensor, velocidade_kmh) VALUES (?, ?)",
    ("Radar-A1", 44.4),
)
con.commit()    # confirma a gravação

# Consulta: ocorrências acima de 40 km/h
for linha in cur.execute(
    "SELECT sensor, velocidade_kmh FROM ocorrencias WHERE velocidade_kmh > 40"
):
    print(linha)

con.close()

Para inserir **vários** registros de uma vez, use `executemany`; e note como a consulta pode ordenar e filtrar sem que escrevamos um único laço.

**Listagem 4.12 — Inserção em lote e consulta ordenada.**

In [ ]:
import sqlite3

registros = [
    ("Radar-B2", 50.0),
    ("Sonar-1", 18.5),
    ("Radar-A1", 61.2),
]

con = sqlite3.connect("ocorrencias.db")
cur = con.cursor()
cur.executemany(
    "INSERT INTO ocorrencias (sensor, velocidade_kmh) VALUES (?, ?)",
    registros,
)
con.commit()

print("Alertas, do mais rápido ao mais lento:")
for sensor, vel in cur.execute(
    "SELECT sensor, velocidade_kmh FROM ocorrencias "
    "WHERE velocidade_kmh > 40 ORDER BY velocidade_kmh DESC"
):
    print(f" - {sensor}: {vel} km/h")

con.close()

> ⚠️ **Armadilha comum** — Nunca monte comandos SQL **concatenando** texto vindo do usuário, como `"... VALUES ('" + nome + "')"`. Essa prática abre a porta para o ataque conhecido como *SQL injection*, em que uma entrada maliciosa altera o comando. Use sempre os **parâmetros** `?`, como nos exemplos: o SQLite cuida de inserir os valores com segurança. Em sistemas de defesa, essa não é uma preocupação acadêmica.

> 📝 **Nota** — Dois passos são fáceis de esquecer e geram confusão. O `con.commit()` **confirma** as alterações — sem ele, as inserções podem se perder. E o `con.close()` encerra a conexão, liberando o arquivo. O bloco `with` também funciona com conexões SQLite, automatizando a confirmação.

## 4.6 Extraindo informação de texto: expressões regulares

Nem todo dado chega organizado em campos. Muitas vezes a informação está embutida em **texto livre** — uma linha de log, um relatório, uma mensagem. As **expressões regulares** (*regex*) são uma linguagem concisa para descrever **padrões** de texto e extrair o que interessa. O módulo `re` as implementa.

**Listagem 4.13 — Extraindo campos de uma linha de log com regex.**

In [ ]:
import re

linha = "2026-05-24 14:32:01 SENSOR=Radar-A1 VEL=44.4 POS=-22.9,-43.2"

# Padrão: SENSOR= seguido de caracteres não-espaço
sensor = re.search(r"SENSOR=(\S+)", linha)
velocidade = re.search(r"VEL=([\d.]+)", linha)

if sensor and velocidade:
    print("Sensor:", sensor.group(1))         # Radar-A1
    print("Velocidade:", velocidade.group(1)) # 44.4

O `r"..."` indica uma **raw string**, em que a barra invertida é tratada literalmente — conveniente para regex. No padrão, `\S+` significa "um ou mais caracteres que não são espaço", e `[\d.]+` significa "um ou mais dígitos ou pontos". Os parênteses definem um **grupo de captura**: a parte que queremos extrair, recuperada com `.group(1)`.

Alguns elementos essenciais da linguagem de padrões:

| Padrão | Casa com |
|---|---|
| `\d` | um dígito (0–9) |
| `\S` | qualquer caractere que não seja espaço |
| `.` | qualquer caractere |
| `+` | uma ou mais repetições do anterior |
| `*` | zero ou mais repetições do anterior |
| `( )` | grupo de captura (o que se quer extrair) |

Para extrair **todas** as ocorrências de um padrão, e não apenas a primeira, usa-se `re.findall`.

**Listagem 4.14 — Extraindo todos os números de um texto.**

In [ ]:
import re

texto = "Detectados 3 contatos a 44.4, 50.0 e 18.5 km/h."
numeros = re.findall(r"[\d.]+", texto)
print(numeros)     # ['3', '44.4', '50.0', '18.5', '.']  <- e o ponto final!

# O padrão aceita "um ou mais dígitos OU pontos" — e o ponto final da frase
# casa sozinho. Exigir ao menos um dígito resolve:
print(re.findall(r"\d[\d.]*", texto))    # ['3', '44.4', '50.0', '18.5']

Esse resultado inesperado é a lição mais útil da seção: uma regex faz **exatamente** o que você escreveu, não o que você quis dizer. Teste sempre com dados reais, e desconfie de padrões que casam com "quase tudo".

Note também o que `re.search` devolve quando **não** encontra o padrão: `None`. Testar esse retorno antes de chamar `.group` é obrigatório — e é aqui que regex e tratamento de erros se encontram.

**Listagem 4.15 — Quando o padrão não casa.**

In [ ]:
import re

linhas = [
    "SENSOR=Radar-A1 VEL=44.4",
    "SENSOR=Radar-B2 VEL=???",     # malformada
    "linha completamente fora do padrão",
]

for linha in linhas:
    achado = re.search(r"VEL=([\d.]+)", linha)
    if achado is None:
        print(f"[descartada] {linha}")
    else:
        print(f"[ok] velocidade = {float(achado.group(1))}")

> ✅ **Boa prática** — Expressões regulares são poderosas, mas tornam-se ilegíveis quando complexas. Para padrões simples de extração, são insuperáveis. Para estruturas mais elaboradas, com muitos campos, prefira um formato estruturado (CSV, JSON) **na origem** dos dados, em vez de tentar decifrar texto livre com uma regex gigantesca. A melhor regex é, muitas vezes, a que você não precisou escrever.

## 4.7 Miniprojeto: dando persistência ao sistema (Módulo 2)

É hora de eliminar a fragilidade apontada ao fim do Capítulo 3. Vamos dotar o miniprojeto de **persistência**: a capacidade de gravar as ocorrências em disco e recuperá-las em uma execução posterior. Adotaremos o **JSON** como formato principal — ele espelha com fidelidade a nossa lista de dicionários — e envolveremos as operações de arquivo em **tratamento de exceções**, para que o sistema resista a arquivos ausentes ou corrompidos.

**Listagem 4.16 — Miniprojeto, Módulo 2: persistência em JSON com tratamento de erros.**

In [ ]:
import json

ARQUIVO = "miniprojeto_ocorrencias.json"

def salvar(ocorrencias, caminho=ARQUIVO):
    """Grava a lista de ocorrências em JSON. Devolve True se deu certo."""
    try:
        with open(caminho, "w", encoding="utf-8") as f:
            json.dump(ocorrencias, f, ensure_ascii=False, indent=2)
        return True
    except OSError as erro:
        print(f"Falha ao salvar: {erro}")
        return False

def carregar(caminho=ARQUIVO):
    """Recupera as ocorrências do arquivo. Se não existir, começa vazio."""
    try:
        with open(caminho, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print("Arquivo ainda não existe; iniciando base vazia.")
        return []
    except json.JSONDecodeError:
        print("Arquivo corrompido; iniciando base vazia.")
        return []

# ---- Uso: recupera, registra uma nova ocorrência e salva ----
ocorrencias = carregar()

ocorrencias.append({
    "id": len(ocorrencias) + 1,
    "sensor": "Radar-A1",
    "velocidade_kmh": 44.4,
    "alerta": True,
})

if salvar(ocorrencias):
    print(f"Base com {len(ocorrencias)} ocorrência(s) gravada com sucesso.")

**Execute a célula acima duas vezes seguidas** e observe: na primeira, ela avisa que o arquivo não existe e cria a base; na segunda, **recupera** a ocorrência gravada antes e acrescenta uma nova. O sistema, enfim, tem memória entre execuções.

Note como cada função trata os erros que lhe são próprios — `salvar` lida com falhas de escrita (`OSError`); `carregar` distingue o arquivo ausente do arquivo corrompido —, sempre devolvendo um resultado utilizável em vez de quebrar. A célula seguinte comprova o segundo caso, corrompendo o arquivo de propósito.

**Listagem 4.17 — Testando a resistência a um arquivo corrompido.**

In [ ]:
# Corrompemos o arquivo de propósito: um JSON inválido
with open(ARQUIVO, "w", encoding="utf-8") as f:
    f.write("{isto não é JSON válido")

base = carregar()          # não quebra: avisa e devolve lista vazia
print("Base recuperada:", base)

# Restaura uma base sadia para as próximas células
salvar([{"id": 1, "sensor": "Radar-A1", "velocidade_kmh": 44.4, "alerta": True}])
print("Base restaurada.")

> 🛡️ **Contexto de defesa** — A persistência confiável é a espinha dorsal de qualquer sistema de registro. Em um contexto operacional, a perda de dados de monitoramento — por um arquivo corrompido, uma gravação interrompida, um erro não tratado — pode significar a perda de evidências ou a interrupção do acompanhamento de uma situação. Projetar a persistência para **falhar de forma segura**, preservando o que já foi gravado, é uma responsabilidade de engenharia.

> 📝 **Nota** — Para volumes maiores, ou consultas mais elaboradas, a mesma persistência pode ser feita em SQLite, com a vantagem das buscas em SQL da Seção 4.5. A escolha entre JSON e banco de dados depende da escala e das consultas necessárias — mais um exemplo de decisão de engenharia. No Módulo 2, o JSON nos basta; o Ex. 4.7 convida a esboçar a versão em banco.

## 4.8 O caminho à frente

O miniprojeto já **modela** e **persiste** seus dados. Mas, à medida que cresce, o código começa a mostrar limites: dados e funções vivem soltos, e manter a coerência entre eles fica cada vez mais trabalhoso. O Capítulo 5 introduz a **programação orientada a objetos** — uma forma de organizar dados e comportamento em unidades coesas, as **classes**. Reestruturaremos o miniprojeto nessa arquitetura, mais limpa e manutenível, completando o Módulo 2. É a transição de um conjunto de *scripts* para um *sistema* de verdade.

## 4.9 Resumo do capítulo
- **Arquivos** de texto são abertos com `open` em modo `r`, `w` ou `a`; o bloco `with` garante o fechamento seguro, e `encoding="utf-8"` preserva os acentos.
- O **tratamento de exceções** (`try`/`except`/`else`/`finally`) captura erros específicos (`FileNotFoundError`, `ValueError`…) e permite a *degradação graciosa* em vez da quebra.
- O **CSV** guarda dados tabulares (`csv.DictReader` e `DictWriter`); lembre-se de que todo valor lido vem como **texto**.
- O **JSON** persiste estruturas (dicionários e listas) com fidelidade — ideal para a lista de ocorrências; use `ensure_ascii=False` e `indent` para legibilidade.
- O **SQLite** é um banco de dados em arquivo, consultável por SQL; use **parâmetros** `?` (nunca concatenação) e lembre-se do `commit`.
- As **expressões regulares** (módulo `re`) extraem padrões de texto livre, com `re.search`, grupos de captura e `re.findall`.
- No **Módulo 2**, o miniprojeto ganhou persistência em JSON com tratamento de erros, gravando e recuperando ocorrências entre execuções.

## Armadilhas comuns
- **Esquecer o `encoding`.** Sem `encoding="utf-8"`, acentos podem ser gravados ou lidos de forma incorreta, com erros que só aparecem em outra máquina.
- **Abrir em modo `"w"` sem querer.** O modo `w` *sobrescreve* o arquivo sem aviso; para acrescentar, use `"a"`.
- **Não converter os dados lidos.** Valores de CSV (e de `input`) chegam como texto; converta com `int` ou `float` antes de calcular.
- **Capturar exceções genéricas.** Um `except:` vazio esconde erros reais; capture o tipo específico que você sabe tratar.
- **Esquecer o `commit` no SQLite.** Sem confirmar, as alterações podem não ser gravadas.
- **Montar SQL por concatenação.** Abre brecha para *SQL injection*; use sempre parâmetros `?`.
- **Chamar `.group()` sem checar `None`.** `re.search` devolve `None` quando não encontra o padrão.

## Exercícios

### Essencial — fixação
**Ex. 4.1** Escreva um programa que grave, em um arquivo `frota.txt`, os nomes de cinco meios — um por linha. Em seguida, leia o arquivo e imprima cada nome precedido de um número de ordem (use `enumerate`).

In [ ]:
# Ex. 4.1
meios = ["Fragata", "Corveta", "Submarino", "Navio-Patrulha", "Rebocador"]

with open("frota.txt", "w", encoding="utf-8") as f:
    for nome in meios:
        f.write(nome + "\n")

with open("frota.txt", "r", encoding="utf-8") as f:
    for ordem, linha in enumerate(f, start=1):
        print(f"{ordem}. {linha.strip()}")

**Ex. 4.2** Peça ao usuário, com `input`, uma velocidade em nós e converta-a para km/h. Envolva a conversão em um `try`/`except` que capture `ValueError` e exiba uma mensagem amigável caso o usuário digite algo que não seja um número.

In [ ]:
# Ex. 4.2
entrada = input("Velocidade em nós: ")
try:
    nos = float(entrada)
except ValueError:
    print(f"'{entrada}' não é um número válido. Informe algo como 24 ou 24.5.")
else:
    print(f"{nos} nós = {nos * 1.852:.1f} km/h")

**Ex. 4.3** Grave a lista de dicionários de ocorrências em `ocorrencias.json` com `json.dump` e, em seguida, leia-a de volta com `json.load`, imprimindo quantas ocorrências foram recuperadas.

In [ ]:
# Ex. 4.3
import json

ocorrencias = [
    {"id": 1, "sensor": "Radar-A1", "velocidade_kmh": 44.4, "alerta": True},
    {"id": 2, "sensor": "Radar-B2", "velocidade_kmh": 50.0, "alerta": True},
    {"id": 3, "sensor": "Sonar-1", "velocidade_kmh": 18.5, "alerta": False},
]

with open("ocorrencias.json", "w", encoding="utf-8") as f:
    json.dump(ocorrencias, f, ensure_ascii=False, indent=2)

with open("ocorrencias.json", "r", encoding="utf-8") as f:
    recuperadas = json.load(f)

print(f"Recuperadas {len(recuperadas)} ocorrências.")

### Tático — aplicação
**Ex. 4.4** Escreva `ler_leituras(caminho)`, que leia um CSV com as colunas `sensor` e `velocidade_kmh` e devolva uma lista de dicionários, **já convertendo** a velocidade para `float`. Trate o caso de o arquivo não existir, devolvendo lista vazia.

In [ ]:
# Ex. 4.4
import csv

def ler_leituras(caminho):
    try:
        with open(caminho, "r", newline="", encoding="utf-8") as f:
            return [
                {"sensor": l["sensor"], "velocidade_kmh": float(l["velocidade_kmh"])}
                for l in csv.DictReader(f)
            ]
    except FileNotFoundError:
        print(f"Arquivo '{caminho}' não encontrado; devolvendo lista vazia.")
        return []

print(ler_leituras("ocorrencias.csv"))   # criado na Listagem 4.7
print(ler_leituras("nao_existe.csv"))    # trata o erro

**Ex. 4.5** Dada uma linha de log no formato `"... SENSOR=Radar-B2 VEL=50.0 ..."`, escreva `extrair(linha)` que use expressões regulares para devolver a tupla `(sensor, velocidade)`, com a velocidade já convertida para `float`. Teste com algumas linhas diferentes.

In [ ]:
# Ex. 4.5
import re

def extrair(linha):
    """Devolve (sensor, velocidade) ou None se a linha for malformada."""
    sensor = re.search(r"SENSOR=(\S+)", linha)
    vel = re.search(r"VEL=([\d.]+)", linha)
    if sensor is None or vel is None:
        return None
    return sensor.group(1), float(vel.group(1))

testes = [
    "2026-05-24 14:32:01 SENSOR=Radar-A1 VEL=44.4 POS=-22.9,-43.2",
    "2026-05-24 14:35:10 SENSOR=Radar-B2 VEL=50.0",
    "2026-05-24 14:36:00 SENSOR=Sonar-1 VEL=??? (falha)",
]
for t in testes:
    print(extrair(t))

**Ex. 4.6** Crie um banco SQLite com uma tabela `meios` (colunas `nome` e `tipo`), insira três meios usando parâmetros `?` e, em seguida, consulte e imprima todos os meios de um tipo específico.

In [ ]:
# Ex. 4.6
import sqlite3

con = sqlite3.connect("frota.db")
cur = con.cursor()
cur.execute("CREATE TABLE IF NOT EXISTS meios (nome TEXT, tipo TEXT)")
cur.execute("DELETE FROM meios")          # limpa, para poder reexecutar a célula

cur.executemany(
    "INSERT INTO meios (nome, tipo) VALUES (?, ?)",
    [("F-1", "fragata"), ("C-1", "corveta"), ("F-2", "fragata")],
)
con.commit()

procurado = "fragata"
for nome, tipo in cur.execute("SELECT nome, tipo FROM meios WHERE tipo = ?", (procurado,)):
    print(f"{nome} ({tipo})")

con.close()

### Estratégico — extensão criativa
**Ex. 4.7** Crie uma versão do miniprojeto que persista em **SQLite** em vez de JSON: `criar_tabela(con)`, `inserir_ocorrencia(con, ocorrencia)` e `listar_alertas(con, limite)`, esta última com uma consulta SQL com `WHERE`. Compare com a versão em JSON da Listagem 4.16: em que situações cada abordagem é preferível?

In [ ]:
# Ex. 4.7 — esqueleto
import sqlite3

def criar_tabela(con):
    con.execute("""
        CREATE TABLE IF NOT EXISTS ocorrencias (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            sensor TEXT NOT NULL,
            velocidade_kmh REAL NOT NULL
        )
    """)
    con.commit()

def inserir_ocorrencia(con, ocorrencia):
    con.execute(
        "INSERT INTO ocorrencias (sensor, velocidade_kmh) VALUES (?, ?)",
        (ocorrencia["sensor"], ocorrencia["velocidade_kmh"]),
    )
    con.commit()

def listar_alertas(con, limite):
    cur = con.execute(
        "SELECT id, sensor, velocidade_kmh FROM ocorrencias "
        "WHERE velocidade_kmh > ? ORDER BY velocidade_kmh DESC",
        (limite,),
    )
    return cur.fetchall()

con = sqlite3.connect("miniprojeto.db")
criar_tabela(con)
for o in [{"sensor": "Radar-A1", "velocidade_kmh": 44.4},
          {"sensor": "Sonar-1", "velocidade_kmh": 18.5}]:
    inserir_ocorrencia(con, o)

for linha in listar_alertas(con, 40.0):
    print(linha)
con.close()

# Reflexão: o JSON é simples e legível, bom para bases pequenas gravadas de uma vez.
# O SQLite vence quando há muitos registros, consultas seletivas e gravações
# incrementais — nele, filtrar não exige carregar tudo na memória.

**Ex. 4.8** Imagine que as ocorrências cheguem como linhas de log em texto livre, em `log.txt`. Escreva um programa que leia o arquivo linha a linha, use regex para extrair sensor e velocidade, **descarte (com tratamento de exceção) as linhas malformadas** e grave o resultado consolidado em `ocorrencias_log.json`. Reflita: esse é, em miniatura, um **pipeline de ingestão de dados** — o primeiro passo de muitos sistemas reais de monitoramento.

In [ ]:
# Ex. 4.8 — esqueleto
import json
import re

# --- 1. Simulamos o arquivo de log (com duas linhas defeituosas de propósito) ---
log = """2026-05-24 14:32:01 SENSOR=Radar-A1 VEL=44.4 POS=-22.9,-43.2
2026-05-24 14:33:12 SENSOR=Radar-B2 VEL=50.0 POS=-23.1,-43.5
2026-05-24 14:34:00 SENSOR=Sonar-1 VEL=??? POS=-23.0,-43.3
linha corrompida sem estrutura alguma
2026-05-24 14:35:47 SENSOR=Radar-A1 VEL=61.2 POS=-22.7,-43.0
"""
with open("log.txt", "w", encoding="utf-8") as f:
    f.write(log)

# --- 2. Ingestão: extrai, valida, descarta o que não presta ---
ocorrencias, descartadas = [], 0

with open("log.txt", "r", encoding="utf-8") as f:
    for numero, linha in enumerate(f, start=1):
        achado = re.search(r"SENSOR=(\S+)\s+VEL=(\S+)", linha)
        if achado is None:
            descartadas += 1
            print(f"[linha {numero}] fora do padrão — descartada")
            continue
        try:
            velocidade = float(achado.group(2))
        except ValueError:
            descartadas += 1
            print(f"[linha {numero}] velocidade inválida ({achado.group(2)!r}) — descartada")
            continue
        ocorrencias.append({
            "id": len(ocorrencias) + 1,
            "sensor": achado.group(1),
            "velocidade_kmh": velocidade,
            "alerta": velocidade > 40.0,
        })

# --- 3. Consolidação ---
with open("ocorrencias_log.json", "w", encoding="utf-8") as f:
    json.dump(ocorrencias, f, ensure_ascii=False, indent=2)

print(f"\n{len(ocorrencias)} ocorrência(s) ingerida(s), {descartadas} descartada(s).")

---

*Fim do Capítulo 4. No Capítulo 5, o miniprojeto será reescrito com **classes** — a transição de scripts para sistema.*